<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12_optional/Ex12.1_39_edge_optional_basin_stability.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.1 (optional, 39 buses) — Basin Stability of a Grid

*Optional and solved: the 39-bus extension of the six-bus Ex_12.1, not required; its report is optional.*

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L12.1 · Power Grid Stability Estimation**

A grid is in step when every machine turns at the same frequency. After a
large disturbance - a fault, a plant lost, a sudden swing of wind power -
does it come back into step, and **from which buses is it easiest to throw
out of step?** As converters replace machines and branches trip, the answer
changes from hour to hour.

You answer it on the IEEE 39-bus grid. First the two conventional studies:
the eigenvalues of the linearised model, and the critical clearing time of a
fault. Then **basin stability**: for every bus, the share of random large
kicks after which the grid returns to step - a colour map of the grid, by
Monte Carlo. Then a **graph network** that estimates the whole map from the
grid's data in a millisecond, and the two compared.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data, its physics, and what the conventional studies say | the model's right-hand side |
| **4** | basin stability by Monte Carlo: the reference and the classical method | — |
| **5** | a graph network for the map, and the comparison | the graph layer |
| **6** | renewable shares and tripped branches the network has not seen | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

Use a GPU runtime (Runtime → Change runtime type → T4 GPU): the timings of sections 4 and 5 are meant for it. On a
CPU it takes about six minutes, most of it Monte Carlo. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem_12_1_39.py', 'grid39.py', 'Ex12.1_39_edge_snbs_maps.npz']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12_optional/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem_12_1_39"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem_12_1_39.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem_12_1_39 as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex12.1_39_edge_optional_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

**The grid.** The IEEE 39-bus system is a reduced model of the New England
grid: 39 buses, 46 branches (lines and transformers), 10 machines. Machine 1
at bus 39 stands for the rest of the North American grid; the other nine are
the local power plants, each behind its step-up transformer. It is the
standard benchmark for stability studies.

**The question.** After a large disturbance, does the grid return to step?
Two studies answer it in a control room today (L12.1's *Conventional
Stability*):

* the **eigenvalues** of the model linearised at the operating point - they
  say whether a *small* disturbance dies out;
* the **critical clearing time** of a fault - the longest a fault can last
  before the protection clears it, found by simulating the fault again and
  again (bisection). One number per fault and per operating point.

**Basin stability** asks a third question, with one answer per bus: kick bus
$i$ hard - its angle anywhere on the circle, its frequency by up to 1 Hz - and
let the grid run. The **single-node basin stability** $S_i$ is the share of
such kicks after which the grid comes back into step (L12.1's *Basin
Stability*). $S_i = 1$: the grid always recovers from bus $i$; $S_i = 0.5$:
half the kicks throw it out of step.

## 2 · The data

| | |
|---|---|
| grid | IEEE 39-bus (New England), 100 MVA base, 50 Hz; network and power flow from MATPOWER's `case39` |
| machines | third order: angle, frequency, voltage behind $x'_d$; $H$, $x_d$, $x'_d$, $T'_{d0}$ of Pai (1989), *to be checked against the book* |
| loads | first order: no inertia; a load bus follows its neighbours within about 10 ms - **assumed**, a frequency-dependent load |
| converters | a renewable plant that replaced a machine: second order, inertia 2 s, $D/M$ = 1 1/s, no field winding - **assumed** |
| scenarios | load 0.9 to 1.1 of the case; at most one of the 35 branches whose loss leaves the grid whole; 0 to 45 % of the nine local machines' power from converters |
| a kick | the angle of one bus moved anywhere in $[-\pi, \pi]$, its frequency (at a machine or a converter) by up to $\pm 1$ Hz |
| back in step | after 10 s, every machine and converter within 0.16 Hz of nominal |

## 3 · The physics

Every bus $i$ has an angle $\delta_i$ (the Kuramoto model of a grid). A
machine or a converter also has a frequency deviation $\omega_i$, and a
machine has the voltage $E_i$ behind its transient reactance, which moves
with its field winding (L12.1's *Third-Order Model*):

$$\frac{d\delta_i}{dt} = \omega_i, \qquad
  M_i\frac{d\omega_i}{dt} = P_i - D_i\,\omega_i - \sum_j E_iE_jB_{ij}\sin(\delta_i-\delta_j), \qquad
  T_i\frac{dE_i}{dt} = E_{f,i} - E_i + X_i\sum_j B_{ij}E_j\cos(\delta_i-\delta_j)$$

with $M_i = 2H_i/\omega_s$ the inertia, $D_i$ the damping, $P_i$ the power
the bus injects (a machine) or draws (a load), $B_{ij}$ the susceptance of
the network between buses $i$ and $j$ (the diagonal included), $E_{f,i}$ the
field voltage, $X_i = x_d - x'_d$ and $T_i = T'_{d0}$. A converter keeps
its voltage ($dE_i/dt = 0$). **A load bus has no inertia**: its angle moves
with its power balance, $D_i\,d\delta_i/dt = P_i - \sum_j E_iE_jB_{ij}\sin(\delta_i-\delta_j)$,
and it keeps its voltage.

The sum over $j$ couples every bus to its neighbours: **the grid is a graph**,
and the sum is two matrix products, $\sin\delta_i\,(B\,E\cos\delta)_i -
\cos\delta_i\,(B\,E\sin\delta)_i$.

**TODO 1** writes the three right-hand sides. The cell after it checks them
against the library's, then computes what the conventional studies say.

In [ ]:
def rhs(g, z):
    # the model's right-hand side for the state z = (delta, omega, E), each (runs, 39)
    d, w, E = z
    p = g.t                                                   # the grid's arrays as tensors: B, P, M, D, X, T, field, Ef, inertial
    s, c = torch.sin(d), torch.cos(d)
    Bs, Bc = (E * s) @ p["B"], (E * c) @ p["B"]               # sum_j B_ij E_j sin(delta_j), and with cos
    pe = E * (s * Bc - c * Bs)                                # the power each bus sends into the network
    q = c * Bc + s * Bs                                       # sum_j B_ij E_j cos(delta_i - delta_j)
    a = p["inertial"]                                         # 1 at a machine or converter, 0 at a load bus
    # TODO 1: return the three right-hand sides of section 3: d(delta)/dt, d(omega)/dt and dE/dt.
    # Where a = 0 (a load bus) d(delta)/dt is (P - pe) / D and d(omega)/dt is zero.
    # p["field"] is 1 at a machine and 0 elsewhere, so that the other voltages stay put.
    #
    #   return (a * w + (1 - a) * (p["P"] - pe) / p["D"],
    #           a * (p["P"] - p["D"] * w - pe) / p["M"],
    #           p["field"] * (p["Ef"] - E + p["X"] * q) / p["T"])
    raise NotImplementedError("Write the three right-hand sides")

In [ ]:
pb.describe_problem()
grid = pb.Grid().to(DEVICE, torch.float32)                   # the base case: the operating point of case39
z = pb.kick(grid, np.arange(8), np.linspace(-3, 3, 8), np.linspace(-6, 6, 8))   # eight kicked states
mine, theirs = rhs(grid, z), pb.rhs(grid, z)
for name, a, b in zip(("d delta/dt", "d omega/dt", "dE/dt"), mine, theirs):
    check(name, a, b, tol=1e-4 * float(b.abs().max() + 1))

lam = pb.small_signal(grid)                                   # the eigenvalues of the linearised model
osc = lam[lam.imag > 1e-3]
slow = osc[osc.imag / (2 * np.pi) < 3]
print(f"  eigenvalues: largest real part {lam.real.max():.2f} 1/s: every small disturbance dies out")
print(f"  the machines swing against each other at {', '.join(f'{f:.2f}' for f in np.sort(slow.imag / (2 * np.pi)))} Hz")

t0 = time.perf_counter()
cct, runs = pb.critical_clearing_time(grid, fault_bus=5, trip=8)   # a fault at bus 6, cleared by tripping branch 5-6
print(f"  a fault at bus 6 cleared by tripping branch 5-6: critical clearing time {1000 * cct:.0f} ms "
      f"({runs} simulations, {time.perf_counter() - t0:.2f} s)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
pb.draw_grid(ax=ax[0], title="IEEE 39-bus: machines numbered, loads small")
(zA, kA), (zB, kB) = [pb.simulate(grid, pb.kick(grid, [30], [a], [2 * np.pi * 0.3]), 10.0, 0.01, f=lambda zz: rhs(grid, zz), keep_every=5)
                      for a in (1.5, 2.0)]
t = np.arange(len(kA)) * 0.05
slip = np.mean([abs(float(st[1][0, 30])) for st in kB[-40:]]) / (2 * np.pi)
print(f"  machine 2 kicked by 2.0 rad: after 8 s it is still {slip:.2f} Hz off on average - out of step")
for k, name, style in ((kA, "kicked by 1.5 rad: back in step", "-"), (kB, "kicked by 2.0 rad: out of step", "--")):
    ax[1].plot(t, [float(s[1][0, 30]) / (2 * np.pi) for s in k], style, lw=2, label=name)
ax[1].set_xlabel("time  [s]"); ax[1].set_ylabel("frequency of machine 2 (bus 31)  [Hz]"); ax[1].legend()
plt.tight_layout(); plt.show()

**What you should see.** `ok` three times: your right-hand side is the
library's. **Every eigenvalue has a negative real part** (the largest
−0.23 1/s) and the machines swing against each other at 0.48 to 1.41 Hz,
the electromechanical modes of the New England grid: the linearised model
says the grid is stable. The fault at bus 6 cleared by tripping branch 5-6
has a critical clearing time of 219 ms, found by 8 simulations.
On the right, machine 2 kicked twice from the same operating point (its
frequency by 0.3 Hz each time): a kick of 1.5 rad dies out, one of 2.0 rad
throws it out of step, and it keeps slipping
against the grid at about 5.42 Hz. **The eigenvalues could not tell these
two apart**: they speak of small disturbances only.

---

## 4 · Basin stability by Monte Carlo

**The method** (L12.1's *Basin Stability*): for every bus, kick it $n$ times
at random, simulate each kick for 10 s, and count the share that comes back
into step. That is $39\,n$ simulations for one map. They do not depend on
each other, so they run as one batch on the GPU: one row per kick, RK4 on a
10 ms step (10 ms changed no verdict against 1 ms in 600 test runs).

**The accuracy is set by $n$.** A share $S$ estimated from $n$ kicks has a
standard error $\sqrt{S(1-S)/n}$, at most $0.5/\sqrt{n}$. **The agreed
accuracy is 0.05**, so the classical method uses $n = 100$ kicks per bus:
3 900 simulations per map.

**The reference** is the same Monte Carlo with $n = 300$ (standard error at
most 0.03), computed once for 35 scenarios and fetched with the libraries.
The cell computes the map of the base case with 100 kicks per bus, and then
the 15 test scenarios the same way, to score the classical method against the
reference.

In [ ]:
maps = pb.load_maps()                                         # the scenarios, and their reference maps
def grid_of(m, i):                                            # scenario i of a group, as a Grid on the device
    g = pb.Grid(float(m["load"][i]), (int(m["out"][i]),), tuple(np.flatnonzero(m["converter"][i])))
    return g.to(DEVICE, torch.float32)                        # the model's arrays as float32 tensors on the device

sync = (lambda: torch.cuda.synchronize()) if DEVICE.type == "cuda" else (lambda: None)
sync(); t0 = time.perf_counter()
base_map = pb.basin_stability(grid, n=100, seed=1, f=lambda zz: rhs(grid, zz))   # 3900 kicked runs, 10 s each
sync(); t_mc = time.perf_counter() - t0
print(f"  base case, 100 kicks per bus: {t_mc:.2f} s for the map ({39 * 100} simulations)")
print(f"  machines' basin stability: " + ", ".join(f"{b + 1}: {base_map[b]:.2f}" for b in pb.GEN_BUS))
weak_loads = [i + 1 for i in np.flatnonzero((base_map < 0.95) & ~pb.IS_GEN)]
print(f"  load buses: lowest {base_map[~pb.IS_GEN].min():.2f}; below 0.95: {weak_loads if weak_loads else 'none'}")

test = maps["test_in"]
mc = np.stack([pb.basin_stability(grid_of(test, i), n=100, seed=100 + i) for i in range(len(test["load"]))])
err_mc = np.abs(mc - test["snbs"])
print(f"  15 test scenarios, Monte Carlo with 100 kicks against the reference (300 kicks):")
print(f"    mean error {err_mc.mean():.3f}, at the machines {err_mc[:, pb.GEN_BUS].mean():.3f}, worst {err_mc.max():.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.6))
pb.draw_grid(base_map, grid, ax=ax[0], label="basin stability", title="base case, 100 kicks per bus")
i = int(np.argmax(test["share"]))                             # the test scenario with the most converters
pb.draw_grid(test["snbs"][i], grid_of(test, i), ax=ax[1], label="basin stability",
             title=f"{100 * test['share'][i]:.0f} % converters (ringed), branch {int(test['out'][i])} out: reference")
plt.tight_layout(); plt.show()

**What you should see.** The base map in about 15 s on a CPU. **Every load
bus is at 1.00**: the network holds a load bus firmly, and any kick there
dies out. **The machines are not**: machine 2 (bus 31) returns from only 0.43
of the kicks and machine 5 (bus 34) from 0.50, the others from 0.62 to 0.72 -
machines on a single transformer at the edge of the grid, the "dead ends" of
Menck et al. (2014). Machine 1 (the rest of the system) and machine 10, with
the most inertia and the slowest field, are at 1.00. On the right, a scenario
with many converters: over the training data a converter's bus averages 0.64
against 0.69 for a machine's, because a converter brings less inertia and no
field winding. Against the reference, Monte Carlo with 100 kicks is off by
0.010 on average (0.035 at the machines) and 0.17 at worst, as its standard
error says.

**What it costs.** One map is 3 900 simulations. A control room wanting the
map for every hour and every branch that might trip needs it 35 times per
hour, and again whenever the renewable share moves.

---

## 5 · A graph network for the map

**The map is a function of the grid's data**: where the machines and the
converters are, how much power each bus moves, how strongly the network holds
it. A graph network takes exactly that and returns one number per bus
(L12.1's *Graph Network for Basin Stability*; Nauck et al. 2022). Each bus is
described by eight numbers - its power $P_i$, its inertia (log), whether it is
a machine or a converter, its voltage, $x_d - x'_d$, $T'_{d0}$, and how
strongly the network holds it ($-B_{ii}$) - and the graph is the network's
susceptance between buses, with a branch that is out simply absent.

**One graph layer** lets every bus combine its own numbers with its
neighbours':

$$H' = \tanh\big(H\,W_{\mathrm{self}} + b + \hat{A}\,H\,W_{\mathrm{neigh}}\big)$$

with $H$ the numbers at the 39 buses (one row per bus), $W_{\mathrm{self}}$
and $W_{\mathrm{neigh}}$ weight matrices shared by all buses, $b$ a bias,
and $\hat{A}$ the susceptance graph with self-loops, scaled by the buses'
degrees. One layer moves information one branch along; five layers reach five
branches. A small dense network then turns each bus's numbers into its basin
stability, between 0 and 1.

**The training data** are 200 scenarios (renewable share up to 45 %, 30 of
the 35 branches out at one time or another), each with its Monte Carlo map of
100 kicks per bus. The loss is the mean squared distance from those maps. It
holds no physics: the physics is in the maps, and in the graph the network is
handed.

**TODO 2** is the graph layer.

In [ ]:
class GraphLayer(torch.nn.Module):                            # a network: layers in __init__, the computation in forward
    def __init__(self, n_in, n_out):
        super().__init__()
        self.own = torch.nn.Linear(n_in, n_out)               # dense layer on a bus's own numbers, with the bias
        self.neighbours = torch.nn.Linear(n_in, n_out, bias=False)   # dense layer on what the neighbours send
    def forward(self, A_hat, H):
        # TODO 2: the graph layer of section 5, before its tanh. A_hat has shape (S, 39, 39), H has shape (S, 39, features).
        #
        #   return self.own(H) + A_hat @ self.neighbours(H)       # own numbers, plus the neighbours' summed along the branches
        raise NotImplementedError("Write the graph layer")

In [ ]:
class MapNet(torch.nn.Module):                                # a network: layers in __init__, the computation in forward
    def __init__(self, hidden=64, depth=5):
        super().__init__()
        self.layers = torch.nn.ModuleList([GraphLayer(pb.N_FEATURE if k == 0 else hidden, hidden) for k in range(depth)])  # five graph layers
        self.head = MLP(n_in=hidden, n_out=1, n_hidden=64, n_layers=2)   # dense tanh network, applied to every bus
    def forward(self, A_hat, H):
        for k, layer in enumerate(self.layers):
            Z = torch.tanh(layer(A_hat, H))                   # one graph layer, then tanh
            H = H + Z if k > 0 else Z                         # a residual connection after the first layer
        return torch.sigmoid(self.head(H)).squeeze(-1)        # a share between 0 and 1 at every bus

X_train = pb.node_features(maps["train"])
mean, spread = X_train.reshape(-1, pb.N_FEATURE).mean(0), X_train.reshape(-1, pb.N_FEATURE).std(0)
spread = np.where(spread > 1e-9, spread, 1.0)
def tensors(m):                                               # a group's graphs and bus numbers, as tensors on the device
    X = (pb.node_features(m) - mean) / spread                 # every number to zero mean and unit spread, on the training data
    as_t = lambda a: torch.as_tensor(a, device=DEVICE, dtype=torch.float32)   # NumPy array -> tensor
    return as_t(pb.adjacency(m)), as_t(X)

A_t, H_t = tensors(maps["train"])
y_t = torch.as_tensor(maps["train"]["snbs"], device=DEVICE, dtype=torch.float32)

set_seed(88)                                                  # fix the random start so a run repeats
net = MapNet().to(DEVICE).float()                             # the network, on the device, in float32
loss_fn = lambda: (net(A_t, H_t) - y_t).pow(2).mean()         # mean squared distance from the Monte Carlo maps
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(net, loss_fn, adam_steps=2000, lbfgs_steps=30, lr=3e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"{parameter_count(net)} weights, {len(y_t)} training maps, trained in {train_time:.0f} s (offline, not counted below)")

In [ ]:
def predict(m):
    with torch.no_grad():                                     # no gradient tracking: evaluation only
        return net(*tensors(m)).cpu().numpy()

A1, H1 = tensors({k: v[:1] for k, v in test.items() if np.ndim(v)})   # one scenario, as a control room would ask
def one_map():
    with torch.no_grad():                                     # no gradient tracking: evaluation only
        net(A1, H1); sync()
t_net = middle(one_map)
pred = predict(test)
err_net = np.abs(pred - test["snbs"])

print(f"{'15 test scenarios, against the reference':44s}{'mean':>8s}{'machines':>10s}{'worst':>8s}{'one map':>12s}")
print(f"  {'Monte Carlo, 100 kicks per bus':42s}{err_mc.mean():8.3f}{err_mc[:, pb.GEN_BUS].mean():10.3f}{err_mc.max():8.2f}{t_mc:10.2f} s")
print(f"  {'graph network':42s}{err_net.mean():8.3f}{err_net[:, pb.GEN_BUS].mean():10.3f}{err_net.max():8.2f}{1000 * t_net:9.2f} ms")
print(f"  the network is {t_mc / t_net:,.0f} times faster for one map, on the same runtime")

fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for a, v, name in zip(ax, (test["snbs"][i], mc[i], pred[i]), ("reference, 300 kicks", "Monte Carlo, 100 kicks", "graph network")):
    pb.draw_grid(v, grid_of(test, i), ax=a, label="basin stability", title=name)
plt.tight_layout(); plt.savefig(cc.output_path("Ex12.1_39_edge_optional_report.png"), dpi=150); plt.show()

**What you should see.** On scenarios it has not seen, the network is off by
0.006 on average and 0.019 at the machines, against
0.010 and 0.035 for Monte Carlo with 100 kicks: **the network is closer to
the reference than the classical method**. It has learned the average over
200 scenarios, which smooths out the sampling noise any one map of 100 kicks
carries. On a CPU one map takes 0.32 ms against about 15 s: about 46 000 times
faster on the same runtime, the training (about 150 s) done offline and not
counted. (On a GPU both are faster and the ratio smaller; run it and see.)
The three maps of the scenario with the most converters look alike.

Read that with care. The network has not solved the model. It has learned how
basin stability varies with the data of a bus and its neighbours, **on the
kinds of scenario it was trained on**.

---

## 6 · Scenarios the network has not seen

**The fair test** is a grid unlike the training data. Two groups were held
back, each with its reference map (300 kicks per bus):

* **more converters**: 10 scenarios with 45 to 70 % of the local power from
  converters, where training stopped at 45 %;
* **a branch never out in training**: 10 scenarios with one of five branches
  out that no training scenario had out.

In [ ]:
print(f"{'held-out groups, against the reference':44s}{'mean':>8s}{'machines':>10s}{'worst':>8s}")
unseen = {}
for key, name in (("test_in", "like the training data"), ("out_re", "45 - 70 % converters"), ("out_line", "a branch never out in training")):
    m = maps[key]; e = np.abs(predict(m) - m["snbs"])
    unseen[key] = (e.mean(), e[:, pb.GEN_BUS].mean(), e.max())
    print(f"  {name:42s}{e.mean():8.3f}{e[:, pb.GEN_BUS].mean():10.3f}{e.max():8.2f}")
m = maps["out_re"]
bias = float(np.mean((predict(m) - m["snbs"])[:, pb.GEN_BUS]))
print(f"  at 45 - 70 % converters the network is {bias:+.3f} off at the machines on average "
      f"({'it says the grid is safer than it is' if bias > 0 else 'it says the grid is less safe than it is'})")

**What you should see.** **Both held-out groups are estimated about as well as
the test scenarios**: 0.019 at the machines with 45 to 70 % converters (0.11
at worst, a bias of +0.004), 0.020 with a branch it never saw out (0.09 at
worst). A branch out is an edge removed, and the graph layer handles a graph
it has not seen. More converters is a smaller step than it sounds in this
model: a converter lowers its bus's basin stability by about 0.05, and the
network has seen converters at every one of the nine places.

**So this test did not find the network's limit.** That is a statement about
these held-out groups, not a guarantee: a change the training data never
showed - a converter that behaves differently, a grid twice the size - could
still break it. MP12.1A-39 is such a test.

---

## 7 · What the notebook says

* **The linearised model said the grid is stable, and it is - for small
  disturbances.** Machine 2 kicked by 2.0 rad fell out of step from the same
  operating point; the eigenvalues cannot see that.
* **The critical clearing time answers one question per fault**, by several
  simulations each. Basin stability answers for every bus at once, by many
  simulations: 3 900 for a map at an accuracy of 0.05.
* **The vulnerable buses are the machines at the edge of the grid**, each on
  one transformer; the load buses, held by the network, are not. Replacing a
  machine by a converter lowers its bus's basin stability, by about 0.05 on
  average in this model.
* **The graph network estimated the map better than Monte Carlo with 100
  kicks** on grids like those it was trained on, in a third of a millisecond
  instead of seconds, the training done offline.
* **A branch it had never seen out, and more converters than in training, did
  not trouble it.** That shows the held-out groups were within its reach, not
  that it has no limit; a grid that differs in kind still goes back to Monte
  Carlo.
* **The physics is in the labels and the graph, not in the loss.** The labels
  are only as good as the model behind them: the loads' and converters'
  inertia and damping are assumed, and the machine data are to be checked
  against Pai (1989).

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex12.1_39_edge_optional_report.md` with the results of section 5 and 6 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L12.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The eigenvalues said every scenario is stable, and machine 2 still fell out of step. What do the eigenvalues, the critical clearing time and basin stability each answer, and what did each cost in this notebook? (-> L12.1 Q1)": """
""",
    "Which buses had the lowest basin stability, and why those? Say what replacing a machine by a converter did to its bus, with your numbers. (-> L12.1 Q2)": """
""",
    "Compare the graph network with Monte Carlo on the test scenarios and on the two held-out groups. Why was the network closer to the reference than Monte Carlo, and what would a held-out test have to contain to find where it fails? (-> L12.1 Q3)": """
""",
    "To conclude: an operator wants the basin-stability map every hour. What would you let the network do, what would you send to Monte Carlo, and how would you know which is which? (-> L12.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| 15 test scenarios, against the reference | mean | machines | worst | one map |", "|---|---|---|---|---|",
            f"| Monte Carlo, 100 kicks per bus | {err_mc.mean():.3f} | {err_mc[:, pb.GEN_BUS].mean():.3f} | {err_mc.max():.2f} | {t_mc:.2f} s |",
            f"| graph network | {err_net.mean():.3f} | {err_net[:, pb.GEN_BUS].mean():.3f} | {err_net.max():.2f} | {1000 * t_net:.2f} ms |",
            "", f"Training: {train_time:.0f} s, offline.",
            "", "| held-out group | mean | machines | worst |", "|---|---|---|---|",
            *[f"| {k} | {v[0]:.3f} | {v[1]:.3f} | {v[2]:.2f} |" for k, v in unseen.items()],
            "", f"Critical clearing time, fault at bus 6 cleared by tripping 5-6: {1000 * cct:.0f} ms ({runs} simulations).",
            "", "Loads' and converters' inertia and damping are assumed; machine data to be checked against Pai (1989)."]
    out = ["# Ex_12.1_39_edge_optional — Basin Stability of a Grid", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex12.1_39_edge_optional_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex12.1_39_edge_optional_report.md into Ex12.1_39_edge_optional_report.pdf, with any figure
# saved as Ex12.1_39_edge_optional_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex12.1_39_edge_optional_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex12.1_39_edge_optional_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex12.1_39_edge_optional_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex12.1_39_edge_optional_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP12.1A-39 · Where to put grid-forming converters

**The problem.** The set showed that a bus loses basin stability when its machine is replaced by
a converter that brings little inertia. A converter can be **grid-forming**
instead: it then holds its own voltage angle and brings inertia and damping
of its own, at a cost. Here six of the nine local machines are converters
(about 66 % of the local power) and two of the six may be made grid-forming.
Which two?

| | |
|---|---|
| grid | the IEEE 39-bus model of the set |
| converters | at buses 31, 32, 33, 35, 36 and 37 (IEEE numbering), grid-following: inertia 2 s, $D/M$ = 1 1/s (assumed) |
| grid-forming | inertia 5 s, $D/M$ = 5 1/s (assumed) |
| choices | the 15 pairs of the six, and none |
| judged by | the mean basin stability of the six converter buses |

**Why a plain PINN is not enough.** Each placement is a different grid. Monte Carlo needs 3 900 simulations for
one map at an accuracy of 0.05; a search over placements on a larger grid
needs thousands of maps.

**What you build.** The set's graph network, extended so that a bus can be grid-forming (a new
input), trained on maps you compute, and used to rank all 15 placements in
one batch. You show where it ranks them correctly and where it does not.

**The ground truth you get.** `MP12.1A-39_truth.npz`, from `tools/miniprojects/ex121_39_edge_optional_truth.py A`: the
basin-stability map of all 39 buses for each of the 16 choices, by Monte
Carlo with 300 kicks per bus (standard error at most 0.03), and the weakest
bus's share from 100, 300 and 900 kicks as the check of the method.

**Worked example.** With no grid-forming converter the six converter buses average 0.63; the
weakest bus is the converter at bus 31 (0.49, 0.50 and 0.53 from 100, 300 and
900 kicks). The best pair, buses 33 and 37, raises the average to 0.72; the
worst, 31 and 32, to 0.66. The best three pairs are within 0.02 of each other,
about twice the Monte Carlo's standard error of a six-bus mean (0.01): the
ranking at the top is no sharper than the data.

**Requirements.**

* Every map within **0.05** of the ground truth on average at the machine and converter buses.
* A pair whose mean at the converter buses is within **0.02** of the best pair's.
* All 15 maps in under **0.1 s**.

**What you hand in.** Your ranking of the 15 pairs against the true one, the pair you would build,
and the placements where your network and the ground truth disagree.

### MP12.1B-39 · The inertia of every machine

**The problem.** As converters replace machines, the inertia left in a grid falls and changes
from hour to hour, and operators need to know it. Here the inertia and the
damping of all nine local machines are hidden, and the data are the
frequency records of PMUs at the ten machines after three branch trips.

| | |
|---|---|
| grid | the IEEE 39-bus model of the set, no converters |
| hidden | $H$ and $D/M$ of the nine local machines, each within ±30 % of the set's values |
| events | branches 2-3, 16-17 and 26-29 tripped with no fault |
| records | each machine's frequency every 20 ms for 10 s, noise 1 mHz |

**Why a plain PINN is not enough.** Inertia and damping both slow a machine's response, and an event far from a
machine says little about it. Which events separate which machines is part
of the answer.

**What you build.** An inverse physics-informed network: the trajectories of all ten machines
with every $H$ and $D/M$ trainable, fitted to the model's equations and the
records. You show which parameters each event identifies.

**The ground truth you get.** `MP12.1B-39_readings.npz` to fit and `MP12.1B-39_truth.npz` with the hidden values,
from `tools/miniprojects/ex121_39_edge_optional_truth.py B`; the records on RK4 steps of 10,
5 and 2.5 ms agree to the printed tolerance.

**Worked example.** A 10 % change of a machine's inertia moves the records by 1.22 to 9.04 mHz;
of its damping, by only 0.15 to 1.30 mHz - below the 1 mHz noise for most
machines. The dampings are the hard half of the problem. The records on RK4
steps of 10 and 5 ms agree with 2.5 ms to 1.16e-02 and 3.27e-04 mHz.

**Requirements.**

* Every inertia within **5 %**.
* Every damping within **20 %**, or a statement of which the records cannot give.
* The grid's total local inertia within **2 %**.

**What you hand in.** The recovered inertias and dampings with uncertainties, the total against the
true one, and the event that tells you least.